<a href="https://colab.research.google.com/github/MiriamSchirmer/online-harm-nlp-ws2627/blob/main/notebooks/02_Loading_an_LLM_Mini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mini notebook: loading an open-source LLM and using it as an annotator

**Applied NLP for Online Harm Detection**, University of Konstanz, WS 2026/27

1. Switch on a GPU and load the model **Qwen2.5-1.5B-Instruct** from Hugging Face.
2. Chat with it.
3. Bring the 10 posts into Colab: upload a file, or mount your Google Drive.
4. Let Qwen label the posts with last week's prompt.
5. Compare Qwen's labels with the reference labels: agreement, kappa, precision, recall and F1.
6. Compare a zero-shot prompt with a few-shot prompt.

✏️ marks the cells where you experiment.

## 0. Switch on the GPU

LLMs run much faster on a graphics card (GPU). Colab lends you one for free:

**Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save**

Do this **before** running anything else (changing the runtime restarts the notebook). Then run the cell below. It should print `True` and `Tesla T4`.

In [ ]:
import torch

print(torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("No GPU: go to Runtime → Change runtime type and select T4 GPU.")

## 1. Load the model

Open-source models are shared on [Hugging Face](https://huggingface.co), a platform for models and datasets. Each model has a name of the form `organisation/model`. We use [`Qwen/Qwen2.5-1.5B-Instruct`](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct) by Alibaba:

- **1.5B**: about 1.5 billion parameters (weights). Small enough to load quickly, large enough to follow instructions.
- **Instruct**: trained to follow instructions and chat (the plain *Base* model only continues text).
- It is free to download and does not need an account.

We need two parts, both from the `transformers` library (already installed in Colab):

- the **tokenizer**, which turns text into tokens (numbers) and back,
- the **model**, which predicts the next token.

The first time, Colab downloads about 3 GB, which takes a minute or two. ☕

**Want a stronger model?** In the cell below, put a `#` in front of the first `model_name` line and remove the `#` in front of the second one. `Qwen3-4B-Instruct` labels more reliably, but it is about 8 GB and needs the T4 GPU.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-1.5B-Instruct"       # default: small and fast
# model_name = "Qwen/Qwen3-4B-Instruct-2507"   # larger (and usually better), needs the T4 GPU

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype="auto",        # use the precision the model was saved in (16 bit)
    device_map="auto",   # put the model on the GPU
)

print("Model loaded on:", model.device)

## 2. Chat with the model

Chat models expect a list of **messages**, each with a `role` (`user`, or `system` for general instructions) and its `content`. The tokenizer's **chat template** turns this list into the exact text format the model was trained on. Then `model.generate()` produces the answer token by token.

We wrap these steps in a small function `ask()` so we can reuse them. With `do_sample=False` the model always picks the most likely next token, so it gives the same answer every time you run it (good for annotation).

In [ ]:
def ask(prompt, max_new_tokens=300):
    messages = [{"role": "user", "content": prompt}] # this line is important, we always need to define a role and content
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    new_tokens = output[0][inputs["input_ids"].shape[1]:]   # keep only the answer, not the prompt
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

print(ask("In two sentences: what is hate speech?"))

### ✏️ Your turn

Ask the model something else.

In [ ]:
print(ask("Write your question here."))

## 3. Bring the data into Colab

The 10 posts from last week are in the file `annotation_posts_session3.csv` (on GitHub in the folder `exercises/`; download it first). Besides `id` and `text`, it has a column `gold_label`: the **reference labels** from the instructor. They are one reading of the definitions, not *the* truth: you may disagree with some of them.

### Option A: upload the file

Run the cell and click **Choose files**. The file lands in Colab's working folder (you can see it in the 📁 folder icon on the left).

⚠️ Uploaded files are deleted when the runtime ends (after you close the notebook or are inactive for a while). Next time you have to upload them again.

In [ ]:
from google.colab import files

uploaded = files.upload()   # opens a "Choose files" button
csv_path = "annotation_posts_session3.csv"

### Option B: mount your Google Drive

"Mounting" connects your Google Drive to Colab like a USB stick: Colab can then read and save files in your Drive, and they stay there after the runtime ends. This is the better option for your projects, where you work with the same data over several weeks.

1. Upload `annotation_posts_session3.csv` to your Google Drive, e.g. into a folder `online_harm`.
2. Run the cell and allow access to your Drive.
3. Your Drive now appears under `/content/drive/MyDrive/`. Adjust the path if your folder has a different name.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
csv_path = "/content/drive/MyDrive/online_harm/annotation_posts_session3.csv"

### Read the file

`pandas` reads the CSV file into a table (a **DataFrame**).

In [ ]:
import pandas as pd

df = pd.read_csv(csv_path)
df

## 4. Qwen as annotator: last week's prompt (zero-shot)

We use exactly the prompt from last week, with the definitions from Vidgen & Derczynski (2020), so that we can compare Qwen with the chatbots you used.

This prompt is **zero-shot**: it gives the model the task and the definitions, but not a single labelled example.

In [ ]:
definition_P = ("Content which directs negativity against individuals, typically through aggression, "
                "insults, intimidation, hostility and trolling, amongst other tactics.")
definition_G = ("Content which directs negativity against a social identity, which is defined in relation "
                "to a particular attribute (e.g. ethnic, racial, religious group membership). It includes use "
                "of dehumanising language, making derogatory, demonising or hostile statements, making threats, "
                "and inciting others to engage in violence, amongst other dangerous communications.")

posts = "\n".join(f"{row.id}. {row.text}" for row in df.itertuples())

prompt = f"""You are annotating social media posts for a research project.
Categories: P = person-directed abuse: {definition_P} G = group-directed abuse: {definition_G} N = neither.
For each of the following 10 posts, answer with exactly one label (P, G or N) according to these definitions. Give one label per post and one short sentence explaining why.
Posts:
{posts}"""

print(prompt)

In [ ]:
answer = ask(prompt, max_new_tokens=800)
print(answer)

### From text to labels

The model answers in free text. To compute anything, we need one label per post. The function `parse_labels()` goes through the answer line by line: a line that starts with a post number gets the first stand-alone `P`, `G` or `N` in that line as its label. We will reuse it in section 6.

Always check the result against the text above: if the model changes its format, this step can go wrong.

In [ ]:
import re

def parse_labels(answer):
    labels = {}
    for line in answer.splitlines():
        line = line.replace("*", "")                       # remove markdown bold
        number = re.match(r"\s*(?:Post\s*)?(\d+)", line)
        label = re.search(r"\b([PGN])\b", line[number.end():]) if number else None
        if number and label and int(number.group(1)) not in labels:
            labels[int(number.group(1))] = label.group(1)
    return df["id"].map(labels)

df["qwen_label"] = parse_labels(answer)
print("Labels found:", df["qwen_label"].notna().sum(), "of", len(df))
df[["id", "text", "gold_label", "qwen_label"]]

## 5. How good is Qwen?

### Agreement

We treat the reference labels as one annotator and Qwen as the other, just like annotators A and B last week:

- **Percent agreement**: in how many posts are the labels the same?
- **Cohen's kappa**: how much better than chance is the agreement? (1 = perfect, 0 = chance level)

With only 10 posts, kappa jumps around a lot: one post more or less changes it considerably.

In [ ]:
from sklearn.metrics import cohen_kappa_score

valid = df.dropna(subset=["qwen_label"])   # in case a label could not be read

agreement = (valid["gold_label"] == valid["qwen_label"]).mean()
kappa = cohen_kappa_score(valid["gold_label"], valid["qwen_label"])

print(f"Percent agreement: {agreement:.0%}")
print(f"Cohen's kappa:     {kappa:.2f}")

### Where do they disagree?

The **confusion matrix** shows which labels get mixed up: rows are the reference labels, columns Qwen's labels. The diagonal counts the agreements.

In [ ]:
print(pd.crosstab(valid["gold_label"], valid["qwen_label"],
                  rownames=["reference"], colnames=["Qwen"]))

valid[valid["gold_label"] != valid["qwen_label"]][["id", "text", "gold_label", "qwen_label"]]

### Precision, recall and F1

Now we treat the reference labels as the **ground truth** and Qwen as a classifier (as on the slides from last week), separately for each class:

- **Precision**: of the posts Qwen labelled G, how many really are G?
- **Recall**: of the posts that really are G, how many did Qwen find?
- **F1**: the harmonic mean of precision and recall.
- **support**: how many posts of this class are in the reference labels.
- **macro avg**: the average over the three classes, each class counting equally.

`zero_division=0` sets a value to 0 instead of giving a warning if, e.g., Qwen never uses one of the labels.

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(valid["gold_label"], valid["qwen_label"],
                            labels=["P", "G", "N"], zero_division=0))

## 6. ✏️ Task: zero-shot vs. few-shot

Does it help to show the model a few labelled examples? A **few-shot** prompt adds some examples to the instructions.

The three examples below are invented and are **not** among the 10 posts (otherwise the model could simply copy the answers).

In [ ]:
examples = """Examples:
"You are a pathetic loser and everyone at work knows it." -> P
"These people breed like rats and are ruining our country." -> G
"The referee made three terrible calls tonight." -> N"""

prompt_few_shot = f"""You are annotating social media posts for a research project.
Categories: P = person-directed abuse: {definition_P} G = group-directed abuse: {definition_G} N = neither.
{examples}
For each of the following 10 posts, answer with exactly one label (P, G or N) according to these definitions. Give one label per post and one short sentence explaining why.
Posts:
{posts}"""

answer_few_shot = ask(prompt_few_shot, max_new_tokens=800)
print(answer_few_shot)

Now we compare both prompts with the reference labels, and with each other.

In [ ]:
from sklearn.metrics import f1_score

df["few_shot_label"] = parse_labels(answer_few_shot)

def evaluate(column):
    ok = df.dropna(subset=[column])
    print(f"{column:16} agreement {(ok['gold_label'] == ok[column]).mean():.0%}   "
          f"kappa {cohen_kappa_score(ok['gold_label'], ok[column]):.2f}   "
          f"macro F1 {f1_score(ok['gold_label'], ok[column], average='macro', zero_division=0):.2f}")

evaluate("qwen_label")       # zero-shot
evaluate("few_shot_label")   # few-shot

changed = df[df["qwen_label"] != df["few_shot_label"]]
print(f"\nThe label changed for {len(changed)} of {len(df)} posts:")
changed[["id", "text", "gold_label", "qwen_label", "few_shot_label"]]

### ✏️ Write your own prompt


In [ ]:
my_prompt = prompt   # start from last week's prompt and change it, e.g. prompt.replace("...", "...")

df["my_label"] = parse_labels(ask(my_prompt, max_new_tokens=800))
evaluate("qwen_label")
evaluate("my_label")
df[["id", "gold_label", "qwen_label", "few_shot_label", "my_label"]]